# CatBoost 최종 모델
ZIP의 최종 CatBoost 실험과 사용자 평균 기준모델 비교를 저장소 경로에 맞게 정리한 공개용 사본입니다.


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
df = pd.read_csv('../data/processed/df_final1.csv')
target = 'Book-Rating'
cat_cols = ['User-ID', 'Book-ID', 'Location_country', 'Publisher']
num_cols = ['Age', 'Year-Of-Publication']
for col in cat_cols:
    df[col] = df[col].fillna('MISSING').astype(str)
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(df[num_cols].median())
train, test = train_test_split(df, test_size=0.2, random_state=42)
features = cat_cols + num_cols
cat_idx = list(range(len(cat_cols)))


In [ ]:
model = CatBoostRegressor(iterations=1389, learning_rate=0.05, depth=8, l2_leaf_reg=1.0, random_seed=42, loss_function='RMSE', verbose=100)
model.fit(Pool(train[features], train[target], cat_features=cat_idx))
pred = model.predict(test[features])
print('RMSE:', root_mean_squared_error(test[target], pred))
print('MAE:', mean_absolute_error(test[target], pred))
print('R2:', r2_score(test[target], pred))


In [ ]:
user_mean = train.groupby('User-ID')[target].mean()
baseline = test['User-ID'].map(user_mean).fillna(train[target].mean())
print('UserMean RMSE:', root_mean_squared_error(test[target], baseline))
print('UserMean MAE:', mean_absolute_error(test[target], baseline))
